In [80]:
# Install/upgrade all required packages
#! pip install --upgrade langchain==0.3.26
#! pip install --upgrade "langchain-core>=0.3.76"
#! pip install --upgrade flask==3.1.1 
#! pip install --upgrade sentence-transformers==4.1.0
#! pip install --upgrade pypdf==5.6.1 
#! pip install --upgrade python-dotenv==1.1.0
#! pip install --upgrade chromadb==0.4.22
#! pip install --upgrade langchain-chroma==0.1.2
#! pip install --upgrade langchain-openai==0.3.24
#! pip install --upgrade langchain-community==0.3.26
# Fix for langchain_core.beta module issue - reinstall langchain-core
# ! pip install --force-reinstall --no-cache-dir "langchain-core>=0.3.76"

In [91]:
# Fix for langchain_core.beta module error
# Create a stub module to satisfy the import requirement
# Run this cell early in your notebook, before importing langchain chains
import sys
import types

# Import langchain_core if available
try:
    import langchain_core
except ImportError:
    print("⚠ langchain_core not installed. Please install it first.")
    langchain_core = None

if langchain_core:
    # Create beta module if it doesn't exist
    if 'langchain_core.beta' not in sys.modules:
        beta_module = types.ModuleType('langchain_core.beta')
        beta_module.__file__ = '<stub>'
        beta_module.__package__ = 'langchain_core.beta'
        beta_module.__path__ = []
        sys.modules['langchain_core.beta'] = beta_module
        setattr(langchain_core, 'beta', beta_module)
        print("✓ Created langchain_core.beta stub module")
    else:
        beta_module = sys.modules['langchain_core.beta']
        if not hasattr(beta_module, '__path__'):
            beta_module.__path__ = []
        print("✓ langchain_core.beta already exists")
    
    # Create runnables submodule as a package
    if 'langchain_core.beta.runnables' not in sys.modules:
        runnables_module = types.ModuleType('langchain_core.beta.runnables')
        runnables_module.__file__ = '<stub>'
        runnables_module.__package__ = 'langchain_core.beta.runnables'
        runnables_module.__path__ = []
        sys.modules['langchain_core.beta.runnables'] = runnables_module
        setattr(beta_module, 'runnables', runnables_module)
        print("✓ Created langchain_core.beta.runnables stub module")
    else:
        runnables_module = sys.modules['langchain_core.beta.runnables']
        if not hasattr(runnables_module, '__path__'):
            runnables_module.__path__ = []
        print("✓ langchain_core.beta.runnables already exists")
    
    # Create context submodule
    if 'langchain_core.beta.runnables.context' not in sys.modules:
        context_module = types.ModuleType('langchain_core.beta.runnables.context')
        context_module.__file__ = '<stub>'
        context_module.__package__ = 'langchain_core.beta.runnables'
        sys.modules['langchain_core.beta.runnables.context'] = context_module
        setattr(runnables_module, 'context', context_module)
        print("✓ Created langchain_core.beta.runnables.context stub module")
    else:
        context_module = sys.modules['langchain_core.beta.runnables.context']
    
    # Add config_with_context function stub
    if not hasattr(context_module, 'config_with_context'):
        def config_with_context(*args, **kwargs):
            """Stub function for config_with_context"""
            # Return a minimal config dict
            return kwargs if kwargs else {}
        context_module.config_with_context = config_with_context
        print("✓ Added config_with_context stub function")
    
    # Verify it works
    try:
        import langchain_core.beta
        import langchain_core.beta.runnables
        import langchain_core.beta.runnables.context
        from langchain_core.beta.runnables.context import config_with_context
        print("✓ All beta modules and config_with_context are now available")
    except ImportError as e:
        print(f"⚠ Warning: Could not verify - {e}")
else:
    print("⚠ Cannot create stubs - langchain_core not available")


✓ langchain_core.beta already exists
✓ langchain_core.beta.runnables already exists
✓ Added config_with_context stub function
✓ All beta modules and config_with_context are now available


In [1]:
%pwd

'/Users/hassanfarid/Documents/Freelance_Projects/PakLegal_ChatBot/Pakistan-Constitution-Chatbot/research'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'/Users/hassanfarid/Documents/Freelance_Projects/PakLegal_ChatBot/Pakistan-Constitution-Chatbot'

In [5]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

/opt/anaconda3/envs/medibot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob = "*.pdf",
        loader_cls = PyPDFLoader
    )
    documents = loader.load()
    return documents

data_path = 'dataset/'

extracted_data = load_pdf_files(data_path)

In [10]:
print(extracted_data[5].page_content)

Islaamic_Sharia_Law.pdf  Page 6 of 604 
 
Preface 
 
 
Sharia is the Arabic word for Islamic law, also known as the Law of Allah. Islam classically draws no distinction 
between religious, and secular life. Hence Sharia covers not only religious rituals, but many aspects of day-to-
day life, politics, economics, banking, business or contract law, and social issues. 
 
There are three major components of Islaamic Sharia Law: 
 
1> 
Quraan  
2> Sunnah  
3> Ijtihaad


In [11]:
print("Length of data(pages):",len(extracted_data)) 

Length of data(pages): 1082


In [13]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only "source" in the metadata and the original page_content.
    """
    
    minimal_docs: List[Document] = []
    
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content = doc.page_content,
                metadata = {"source": src}
            )
        )
    return minimal_docs

minimal_docs = filter_to_minimal_docs(extracted_data)

In [14]:
minimal_docs[0]

Document(metadata={'source': 'dataset/islamic-sharia-law.pdf'}, page_content="Islaamic_Sharia_Law.pdf  Page 1 of 604 \n \nafii62822/afii62785/afii62761  \nIn the N ame of Allah, M ost G racious, M ost M erciful \nIslaamic Sharia Law  \nBased on Quraan, Sunnah and Ijtihaad  \nby Irshad Mahmood - Global Auliyaa (PRESIDENT), Siraat-al-Mustaqeem Dawah Centre \nSiraat - al - Mustaqeem Dawah Centre \n \n \nA need for every single Muslim \n \nhttp://global-right-path.net16.net  \nFor Free Download then Forward to as many as you cahttp:// Islaamic_Sharia_Law.pdf \nLaws are NOT  meant to be taken in one's own hand BUT  meant for the state to establish  \n \n \nAllah is One  <---> Quraan is One <---> Siraat_ul_Mustaqeem is One \nOne Ummah <---> One Islaam     <---> One Voice <---> One Worshipping Method  \nRevised-2009 Islaamic Sharia Law for the New Global  World \n \nThose Rulers, Politicians or other persons having Authority, who don't take any interest on establishing \nIslaamic Sharia Law, 

In [15]:
def text_split(docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 20
    )
    text_chunks = text_splitter.split_documents(docs)
    return text_chunks

text_chunks = text_split(minimal_docs)

In [16]:
text_chunks = text_split(minimal_docs)
print(f'Number of Chunks: {len(text_chunks)}')

Number of Chunks: 8482


In [ ]:
# ! pip install tf-keras

In [19]:
from langchain.embeddings import HuggingFaceEmbeddings


def download_embeddings():
    """
    Download and return huggingface embedding model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name
    )
    return embeddings
    
embedding = download_embeddings()

In [34]:
from langchain.embeddings import HuggingFaceEmbeddings

# Use multilingual-e5-large model for better multilingual support
embedding = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={'device': 'mps'},  # Use 'cuda' if you have GPU
    encode_kwargs={'normalize_embeddings': True}  # Normalize embeddings for better cosine similarity
)
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: XLMRobertaModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='intfloat/multilingual-e5-large', cache_folder=None, model_kwargs={'device': 'mps'}, encode_kwargs={'normalize_embeddings': True}, multi_process=False, show_progress=False)

In [35]:
print(embedding)
vectors = embedding.embed_query("Hello, World")

print(f"Vectors (first 10): {vectors[:10]}")
print(f"Length of Vectors: {len(vectors)}")
print(f"✓ Embedding model loaded: intfloat/multilingual-e5-large")

client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: XLMRobertaModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='intfloat/multilingual-e5-large' cache_folder=None model_kwargs={'device': 'mps'} encode_kwargs={'normalize_embeddings': True} multi_process=False show_progress=False
Vectors (first 10): [0.031024456024169922, -0.019993258640170097, -0.0157387126237154, -0.04249153286218643, 0.021823687478899956, -0.01643270067870617, -0.02338492125272751, 0.042664408683776855, 0.05870877951383591, -0.026659149676561356]
Length of Vectors: 1024
✓ Embedding model loaded: intfloat/multilingual-e5-large


In [47]:
from dotenv import load_dotenv
from pathlib import Path

# Load .env file from research/ directory (where the notebook is located)
# Since we changed to parent directory in Cell 2, we need to go back to research/
env_path = Path.cwd() / 'research' / '.env'

# Load .env file
load_dotenv(dotenv_path=env_path)

# ChromaDB doesn't require an API key - it runs locally
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Only set environment variables if they are not None
if OPENAI_API_KEY is not None:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
else:
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file in the research/ directory.")

print("✓ API keys loaded successfully")

✓ API keys loaded successfully


In [48]:
# ChromaDB doesn't require initialization - it's handled automatically by the vector store
# No API keys or external services needed - ChromaDB runs locally
print("✓ ChromaDB will be initialized automatically when creating the vector store")

✓ ChromaDB will be initialized automatically when creating the vector store


In [49]:
# ChromaDB automatically creates the collection when we use from_documents
# No need to manually create an index - ChromaDB handles this automatically
collection_name = "constitution-chatbot"
print(f"✓ ChromaDB collection '{collection_name}' will be created automatically")

✓ ChromaDB collection 'constitution-chatbot' will be created automatically


In [54]:
from langchain_chroma import Chroma

# ChromaDB runs locally and doesn't have the same size restrictions as Pinecone
# It will automatically handle all documents efficiently
doc_search = Chroma.from_documents(
    documents=text_chunks,
    embedding=embedding,
    collection_name=collection_name,
    persist_directory="./chroma_db" 
)

In [59]:
retriever = doc_search.as_retriever(search_type='similarity', search_kwargs={"k":5})

In [60]:
retriev_docs = retriever.invoke("What fundamental rights are guaranteed to citizens under the Constitution of Pakistan?")

In [62]:
type(retriev_docs)
print(retriev_docs[1].page_content)

CONSTITUTION OF PAKISTAN  
 15 
25A. Equality of citizens 
 
25. (1) All citizens are equal before law and are entitled to eq ual 
protection of law. 
 
(2) There shall be no discrimination on the basis of sex 1*. 
 
(3) Nothing in this Article shall prevent the State from making 
any special provision for the protection of women and children. 
 
25. Right to education 
 
2[25A. The State shall provide free and compulsory education to all


In [100]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(
    model = 'gpt-4o'  # Changed from 'GPT-4-turbo' - use lowercase model names
)


In [101]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [106]:
system_prompt = (
    "You are a law-focused assistant specializing in the Constitution of the Islamic Republic of Pakistan. "
    "Your task is to answer questions in a way that is detailed, clear, and easy to understand, even for people "
    "who have no background in law. Always base your responses strictly on the retrieved context provided.\n\n"
    "When answering a question:\n"
    "- Explain the concept in simple, beginner-friendly language without losing legal accuracy.\n"
    "- Provide a detailed explanation, breaking down complex legal terms into plain words.\n"
    "- Identify and cite the most relevant constitutional Article(s) and Part(s).\n"
    "- Where helpful, give short examples or analogies to make the explanation easier.\n"
    "- If the context does not provide enough information, reply only with: 'I don't know.'\n\n"
    "Format your response as:\n"
    "Answer:\n<Your detailed and beginner-friendly explanation here>\n\n"
    "Relevant Article(s): <List the article numbers if available, else 'Not found'>\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [107]:
question_answere_chain = create_stuff_documents_chain(chat_model,prompt)
rag_chain = create_retrieval_chain(retriever, question_answere_chain)

In [108]:

response = rag_chain.invoke({"input": 'Police arrest me on the charges of fake blasphemy case, how can I secure myself?'})
response

{'input': 'Police arrest me on the charges of fake blasphemy case, how can I secure myself?',
 'context': [Document(id='89b9b748-fdf1-48cd-942d-ca36a0b88254', metadata={'source': 'dataset/Pakistan Penal Code (1860).pdf', 'text': 'unlawful purpose shall be punishable with imprisonment for life.] \n \n5[295C. Use of derogatory remarks, etc., in respect of the Holy Prophet. Whoever by \nwords, either spoken or written, or by visible representation, or by any imputation, innuendo, or \ninsinuation, directly or indirectly, defiles the sacred name of the Holy Prophet Muhammad (peace be \nupon him) shall be punished with death, *or imprisonment for life, and shall also be liable to fine.]'}, page_content='unlawful purpose shall be punishable with imprisonment for life.] \n \n5[295C. Use of derogatory remarks, etc., in respect of the Holy Prophet. Whoever by \nwords, either spoken or written, or by visible representation, or by any imputation, innuendo, or \ninsinuation, directly or indirectly

In [109]:
print(response['answer'])

Answer:
Facing charges related to blasphemy, especially if they are false, can be extremely serious and concerning due to the severe penalties involved, such as life imprisonment or even the death penalty. Here are some steps you can consider taking to secure yourself:

1. **Legal Representation:** Immediately seek legal counsel. It's crucial to have an experienced lawyer who understands the complexities of blasphemy laws in Pakistan, specifically under Article 295C of the Constitution or the Penal Code, which deals with derogatory remarks against the Holy Prophet Muhammad (peace be upon him).

2. **Evidence Collection:** Work with your legal team to gather evidence that can support your defense. This might include alibis, witnesses, or contradictions in the accusations against you.

3. **Witness Testimonies:** Identify and present any witnesses who can testify on your behalf, providing evidence that the charges are false.

4. **Media and Advocacy Groups:** In some cases, reaching out 